# Step 7 — Create and inspect an AAPL headline sample

Before processing thousands of headlines, we create a reproducible 200-headline sample. The original files in `data/raw/` remain unchanged.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "apple_news_data.csv"
SAMPLE_PATH = PROJECT_ROOT / "data" / "processed" / "apple_headline_validation_sample.csv"
news = pd.read_csv(RAW_PATH)
news["published_at"] = pd.to_datetime(news["date"], utc=True, errors="coerce")
print(f"Loaded {len(news):,} source rows")

Loaded 29,752 source rows


## Initial relevance rules

A headline qualifies when it:

1. was published during 2023 or 2024;
2. has `AAPL.US` as an exact member of its symbol list;
3. explicitly mentions Apple, AAPL, iPhone, iPad, Mac, iOS, or Vision Pro in the title; and
4. is not a duplicate after normalizing capitalization and whitespace.

This strict first-pass filter favors relevance over completeness. We can broaden it later if manual inspection shows it is too restrictive.

In [2]:
in_period = news["published_at"].between("2023-01-01", "2025-01-01", inclusive="left")
has_aapl_symbol = news["symbols"].fillna("").str.split(",").apply(
    lambda symbols: any(symbol.strip() == "AAPL.US" for symbol in symbols)
)
apple_title_pattern = r"\b(?:Apple|AAPL|iPhone|iPad|Mac|iOS|Vision Pro)\b"
explicit_apple_title = news["title"].str.contains(
    apple_title_pattern, case=False, regex=True, na=False
)

eligible = news.loc[in_period & has_aapl_symbol & explicit_apple_title].copy()
eligible["normalized_title"] = (
    eligible["title"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)
eligible = eligible.sort_values("published_at").drop_duplicates("normalized_title", keep="first")

print(f"Eligible unique headlines: {len(eligible):,}")
eligible["published_at"].dt.year.value_counts().sort_index()

Eligible unique headlines: 5,677


published_at
2023    3502
2024    2175
Name: count, dtype: int64

## Balanced validation sample

We randomly select 100 headlines from each year using a fixed seed. Running the notebook again produces the same sample.

In [3]:
eligible["year"] = eligible["published_at"].dt.year
sample = (
    eligible.groupby("year", group_keys=False)
    .sample(n=100, random_state=42)
    .sort_values("published_at")
)

sample_columns = [
    "published_at", "title", "link", "symbols",
    "sentiment_polarity", "sentiment_neg", "sentiment_neu", "sentiment_pos",
]
sample[sample_columns].to_csv(SAMPLE_PATH, index=False)
print(f"Saved {len(sample)} headlines to {SAMPLE_PATH}")
sample.groupby("year").size()

Saved 200 headlines to /Users/keishakalra/Desktop/Financial_App/data/processed/apple_headline_validation_sample.csv


year
2023    100
2024    100
dtype: int64

## Manual inspection view

Read the titles below and look for irrelevant stories, duplicate ideas, unclear wording, or sentiment scores that seem inconsistent with the language.

In [4]:
pd.set_option("display.max_colwidth", 140)
sample[["published_at", "title", "sentiment_polarity"]].head(30)

,published_at,title,sentiment_polarity
12870,2023-01-03 20:59:30+00:00,More than $1 trillion wiped off value of Apple in face of China chaos,-0.883
12858,2023-01-04 10:09:00+00:00,Apple Stock Poised to Rebound After Downgrade on iPhone Concerns,0.340
12821,2023-01-05 02:00:43+00:00,UPDATE 1-Luxshare says client cooperation normal after Apple production cut report,-0.710
12777,2023-01-07 10:50:00+00:00,3 Compelling Reasons Why Amazon Stock Is a Smarter Pick Than Apple in 2023,0.743
12665,2023-01-11 22:02:00+00:00,Apple could bring once-abhorred touchscreens to computers in 2025: report,0.026
12564,2023-01-17 16:06:53+00:00,"Starbucks Taps Online Ordering Market Via iOS, Android Devices By Extending Collaboration With DoorDash",0.989
12549,2023-01-17 23:16:32+00:00,Apple Introduces Faster MacBook Pros and Mac Minis,0.985
12389,2023-01-26 20:51:15+00:00,Apple iPhone shipments declined almost 15% year-over-year in 2022,-0.382
12298,2023-01-30 21:49:58+00:00,"Apple Profit Could Fall on China Production, Demand Slowdown",0.318
12169,2023-02-02 23:51:04+00:00,Apple earnings: What to watch from the iPhone maker in the quarter ahead,0.000


In [5]:
pd.Series({
    "Sample rows": len(sample),
    "Unique titles": sample["normalized_title"].nunique(),
    "Missing titles": sample["title"].isna().sum(),
    "Missing publication times": sample["published_at"].isna().sum(),
    "Missing sentiment scores": sample["sentiment_polarity"].isna().sum(),
})

Sample rows                  200
Unique titles                200
Missing titles                 0
Missing publication times      0
Missing sentiment scores       0
dtype: int64